In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [4]:
loader=PyPDFLoader("../data/System-Design-Interview-Questions.pdf")
docs=loader.load()

In [5]:
len(docs)

101

In [6]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs=splitter.split_documents(docs)


In [7]:
len(splitted_docs)

158

In [ ]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

### Agent= TOOLs, LLM, Prompt

In [20]:
@tool
def retriever_tool(query:str):
    """
    This tool can help you to retrieve the relevant data of the pdf Documents, and these documents have details about medical reports  
    """
    docs=vector_store.similarity_search(query=query, k=4)
    context=" "
    for doc in docs:
        context=doc.page_content + "\n\n"
        
    return context

In [ ]:
retriever_tool.invoke("What is file system storage")

In [17]:
llm=ChatOpenAI(model="gpt-5")

In [22]:
System_Prompt="""  
You are a helpful assistant that answers questions using retrieved context.
Always use the 'retriever_tool' for questions requiring external knowledge.
"""

In [23]:
agent=create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [ ]:
query="What is file system storage"
response=agent.invoke({"messages":[{"role":"user","content":query}]})   
result=response["messages"][-1].content

In [ ]:
print(result)